In [9]:
import numpy as np
import pandas as pd
import torch

In [6]:
path = "/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/amber/amber_llava_label_with_evidence_and_attn_detection_05_11_2025.pkl"

In [7]:
res_data = pd.read_pickle(path)

In [10]:
res_data.head(4)

,question,answer,question_id,image_id,image_path,labels_with_evidence
0,Describe this image.,The image depicts a group of four people walki...,1,AMBER_1.jpg,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/amb...,"[{'word': 'group', 'evidence': [0.0012797663, ..."
1,Describe this image.,The image features a man wearing a life jacket...,2,AMBER_2.jpg,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/amb...,"[{'word': 'man', 'evidence': [0.008729229, 0.0..."
2,Describe this image.,The image features a young child standing in a...,3,AMBER_3.jpg,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/amb...,"[{'word': 'young', 'evidence': [0.17943619, 0...."
3,Describe this image.,The image features a woman wearing a white shi...,4,AMBER_4.jpg,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/amb...,"[{'word': 'woman', 'evidence': [0.1307752, 0.0..."


In [91]:

all_seqs = []
ids = []
for inx, row in res_data.iterrows():
    res = row["labels_with_evidence"]
    seq = row["answer"]
    hal_words = []
    for i in res:
        viz_evi = (torch.tensor(i["evidence"]) >= 0.37).int().sum().item()
        prob = i["label"]
        # if prob <= 0.7:
        #     hal_words.append(i["word"])
        if viz_evi <= 3 and prob <= 0.99:
            hal_words.append(i["word"])
    for hw in hal_words:
            seq = seq.replace(hw.strip().strip("."), "")
    all_seqs.append(seq)
    ids.append(row["question_id"])

In [92]:
req_df = pd.DataFrame({"id": ids, "response": all_seqs})

In [93]:
req_df.to_json("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/amber/amber_post_halu_res.json", lines=True, orient="records")